# UAE Mobile Intelligence - Peer-Group Composite Classifier

The brief is explicit and mandatory here: **every** peer comparison the product makes (every
"peer median" shown on screen, and later every Peer Gap score) must be against an appropriate
peer group, never the UAE-wide distribution. It is equally explicit about *how not to build it*:
classifying by OSM land-use tag collapses commercial/retail to ~17 zones nationally -- UAE
land-use tagging is simply too thin. Instead, build peer groups from a **composite of population
density, building-footprint density, POI density and road density** -- all already sitting in
`zone_quarter_table.parquet` -- which the brief says yields four stable groups: commercial/
urban-core, low-density residential, industrial, rural/edge.

This notebook builds that classifier with KMeans (`k=4`, matching the brief's four named groups),
inspects the resulting clusters to assign the human-readable labels, and validates the two things
the brief actually cares about: **groups large enough for a robust median**, and **enough within-
UAE spread that real peer gaps exist to find** -- not classification accuracy against some ground
truth, because no ground truth exists here.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Build the per-zone feature table

Density features are static (population and OSM don't change quarter to quarter), so this
classifier runs on one row per unique H3 cell, not per zone-quarter. Population density is
derived here (`population / zone_area_km2`) since the master table only stores the raw count.

`building_count_per_km2` is dropped from the feature set: it correlates 0.79 with
`building_footprint_pct` on this data (both measure "how built-up," one by count, one by
coverage area), so keeping both would double-count the same signal in the clustering distance.
Footprint percentage is kept as the more informative of the two -- a few large warehouses and
many small houses can have the same count but very different footprint.

In [2]:
zone_quarter = pd.read_parquet("../data/processed/zone_quarter_table.parquet")
# Idempotency: this notebook may already have run once and written peer_group back into this
# same file, so drop any prior peer_group before recomputing rather than colliding on re-run.
zone_quarter = zone_quarter.drop(columns=["peer_group"], errors="ignore")
zones = zone_quarter.drop_duplicates("h3_cell").copy()
zones["pop_density_per_km2"] = zones["population"] / zones["zone_area_km2"]

FEATURES = ["pop_density_per_km2", "building_footprint_pct", "poi_count_per_km2", "road_density_km_per_km2"]

print("Zones to classify:", len(zones))
zones[FEATURES].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(2)

Zones to classify: 3400


,pop_density_per_km2,building_footprint_pct,poi_count_per_km2,road_density_km_per_km2
count,3400.00,3400.00,3400.00,3400.00
mean,657.97,1.34,2.84,5.80
std,1099.68,3.98,17.03,6.88
min,0.00,0.00,0.00,0.00
10%,1.57,0.00,0.00,0.10
25%,30.08,0.00,0.00,1.16
50%,193.85,0.00,0.00,3.32
75%,745.78,0.38,0.43,7.55
90%,1961.72,3.34,2.80,15.43
99%,5296.47,20.92,61.88,30.65


## 2. Log-transform, then standardize

All four features are heavily right-skewed -- a lot of near-zero zones and a long tail of dense
ones (e.g. population density spans 0 to ~6,900/km2, POI density 0 to ~400/km2). Without a log
transform, KMeans' Euclidean distance would be dominated entirely by the few extreme zones.
`log1p` handles the many exact-zero zones cleanly (rural cells with literally no OSM features).
Standardizing after that puts all four features on the same scale, since population density
(per km2) and footprint percentage live on completely different numeric ranges.

In [3]:
X_log = np.log1p(zones[FEATURES])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("Feature matrix:", X_scaled.shape)
pd.DataFrame(X_scaled, columns=FEATURES).describe().round(2)

Feature matrix: (3400, 4)


,pop_density_per_km2,building_footprint_pct,poi_count_per_km2,road_density_km_per_km2
count,3400.00,3400.00,3400.00,3400.00
mean,0.00,-0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00
min,-2.07,-0.50,-0.46,-1.58
25%,-0.60,-0.50,-0.46,-0.76
50%,0.19,-0.50,-0.46,-0.02
75%,0.76,-0.08,-0.04,0.71
max,1.71,4.55,6.57,2.54


## 3. KMeans, k=4 -- matching the brief's four named groups

`k=4` isn't tuned by elbow/silhouette search here: the brief specifies four groups by name
(commercial/urban-core, low-density residential, industrial, rural/edge) as the peer-group
structure the product needs, so the target `k` is a requirement, not a free hyperparameter. What
*is* validated below is whether those four clusters are actually usable -- big enough, and
spread out enough -- not whether four is the "best" k in isolation.

In [4]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
zones["cluster"] = kmeans.fit_predict(X_scaled)

print(zones["cluster"].value_counts().sort_index())

cluster
0     830
1     303
2    1413
3     854
Name: count, dtype: int64


## 4. Assign human-readable labels from the cluster centroids

The clustering itself doesn't know what "industrial" means -- labels are assigned afterward from
each cluster's centroid, using an explainable rule so the mapping can be checked, not guessed:

1. Rank clusters by overall density intensity (mean of the four standardized features). The
   highest becomes **commercial/urban-core**, the lowest becomes **rural/edge**.
2. Of the two remaining (middle-intensity) clusters, the one with the *lower* POI density
   relative to its building/road density is **industrial** -- warehouses and depots have
   buildings and roads but comparatively few points of interest (shops, restaurants, amenities);
   the other is **low-density residential**.

In [5]:
# Two inverses needed to get back to real units: undo StandardScaler, then undo log1p.
# Skipping the expm1 step would leave centroids in log-space -- technically fine for ranking
# (monotonic), but meaningless to read (e.g. a population-density centroid of "7.95" when the
# real range is 0-6,900/km2) if anyone asks "why is this zone's peer group X?" later.
centroids_log = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=FEATURES)
centroids = np.expm1(centroids_log)
centroids_std = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURES)  # standardized space, for ranking
centroids["intensity"] = centroids_std.mean(axis=1)
centroids["poi_to_built_ratio"] = centroids["poi_count_per_km2"] / (
    centroids["building_footprint_pct"] + centroids["road_density_km_per_km2"] + 1e-6
)

order = centroids["intensity"].sort_values(ascending=False).index.tolist()
urban_core_cluster, rural_cluster = order[0], order[-1]
middle_clusters = order[1:3]

middle_by_poi_ratio = centroids.loc[middle_clusters, "poi_to_built_ratio"].sort_values()
industrial_cluster = middle_by_poi_ratio.index[0]
residential_cluster = middle_by_poi_ratio.index[1]

LABELS = {
    urban_core_cluster: "commercial/urban-core",
    residential_cluster: "low-density residential",
    industrial_cluster: "industrial",
    rural_cluster: "rural/edge",
}
zones["peer_group"] = zones["cluster"].map(LABELS)

print("Cluster -> label mapping:", LABELS)
print()
centroids.index.name = "cluster"
centroids.round(2)

Cluster -> label mapping: {1: 'commercial/urban-core', 3: 'low-density residential', 2: 'industrial', 0: 'rural/edge'}



,pop_density_per_km2,building_footprint_pct,poi_count_per_km2,road_density_km_per_km2,intensity,poi_to_built_ratio
cluster,,,,,,
0,3.68,0.01,0.03,0.55,-0.86,0.05
1,2822.57,9.73,12.37,20.40,2.07,0.41
2,137.72,0.07,0.09,2.55,-0.24,0.03
3,860.11,0.71,0.63,8.84,0.50,0.07


## 5. Validate -- group sizes and centroid separation

The brief's own failure case (land-use classification collapsing to ~17 commercial zones) is
exactly what this check is for: a group that small can't support a robust median. Centroid
values are printed by label so the ordering can be sanity-checked by eye (urban-core should be
highest on every density feature, rural/edge lowest).

In [6]:
group_sizes = zones["peer_group"].value_counts()
print("Peer-group sizes (zones):")
print(group_sizes.to_string())
print()
print("Smallest group:", group_sizes.idxmin(), "--", group_sizes.min(), "zones")
assert group_sizes.min() >= 30, "A peer group this small can't support a robust median"

print()
print("Centroids by label (original units):")
readable = centroids.rename(index=LABELS).drop(columns=["intensity", "poi_to_built_ratio"])
print(readable.loc[["commercial/urban-core", "low-density residential", "industrial", "rural/edge"]].round(2).to_string())

Peer-group sizes (zones):
peer_group
industrial                 1413
low-density residential     854
rural/edge                  830
commercial/urban-core       303

Smallest group: commercial/urban-core -- 303 zones

Centroids by label (original units):
                         pop_density_per_km2  building_footprint_pct  poi_count_per_km2  road_density_km_per_km2
cluster                                                                                                         
commercial/urban-core                2822.57                    9.73              12.37                    20.40
low-density residential               860.11                    0.71               0.63                     8.84
industrial                            137.72                    0.07               0.09                     2.55
rural/edge                              3.68                    0.01               0.03                     0.55


## 6. Validate -- do real peer gaps exist within each group?

Group medians only matter if there's genuine within-group spread to compare a zone against --
if every zone in a group scores nearly identically, "peer gap" is meaningless. This checks raw
`download_mbps` (test-weighted mean across the zone's quarters) as a proxy, since the Experience
Index itself hasn't been computed yet (next notebook). The brief's own T0 finding -- a roughly
threefold spread between the strongest and weakest deciles nationally -- is the benchmark: each
peer group should show real internal spread, not be artificially uniform.

In [7]:
zone_download = zone_quarter.groupby("h3_cell").apply(
    lambda g: np.average(g["download_mbps"], weights=g["tests"]), include_groups=False
).rename("avg_download_mbps")
zones = zones.join(zone_download, on="h3_cell")

spread = zones.groupby("peer_group")["avg_download_mbps"].describe(percentiles=[.1, .5, .9])[
    ["count", "10%", "50%", "90%"]
]
spread["p90_p10_ratio"] = spread["90%"] / spread["10%"]
print(spread.round(1).to_string())
print()
print("Each group shows real internal spread (p90/p10 ratio well above 1x), confirming peer gaps")
print("are findable within every group, not just across the national distribution.")

                          count    10%    50%    90%  p90_p10_ratio
peer_group                                                         
commercial/urban-core     303.0  289.4  423.5  634.4            2.2
industrial               1413.0   35.3  296.6  759.1           21.5
low-density residential   854.0  151.4  411.6  680.9            4.5
rural/edge                830.0    7.6  113.7  664.0           87.1

Each group shows real internal spread (p90/p10 ratio well above 1x), confirming peer gaps
are findable within every group, not just across the national distribution.


## 7. Save

`peer_groups_uae.parquet` is the standalone static classification (one row per H3 cell). The
label is also merged back into `zone_quarter_table.parquet` so every downstream notebook gets
`peer_group` for free without re-running this classifier.

In [8]:
peer_groups = zones[["h3_cell", "peer_group"] + FEATURES].copy()
out_path = Path("../data/processed/peer_groups_uae.parquet")
peer_groups.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")

zone_quarter_with_peers = zone_quarter.merge(peer_groups[["h3_cell", "peer_group"]], on="h3_cell", how="left")
assert len(zone_quarter_with_peers) == len(zone_quarter), "Merge must not add or drop rows"
assert zone_quarter_with_peers["peer_group"].isna().sum() == 0, "Every measured zone must get a peer group"

out_path2 = Path("../data/processed/zone_quarter_table.parquet")
zone_quarter_with_peers.to_parquet(out_path2, index=False)
print(f"Updated: {out_path2} (+peer_group column, {len(zone_quarter_with_peers)} rows)")

Saved: ..\data\processed\peer_groups_uae.parquet (117.5 KB)
Updated: ..\data\processed\zone_quarter_table.parquet (+peer_group column, 13597 rows)


## Summary

- `data/processed/peer_groups_uae.parquet` -- one row per H3 res-7 cell (3,400 zones), with
  `peer_group` (one of the four brief-mandated labels) and the density features used to assign it.
- `data/processed/zone_quarter_table.parquet` -- now carries `peer_group` on every zone-quarter row.
- All four groups clear a 30-zone minimum size and show real internal spread in raw download
  speed (proxy for Experience Index, not yet computed) -- both the brief's stated risks
  (collapsed group size, no findable within-group gap) are checked and pass.
- Labels are assigned from cluster centroids via an explainable rule (density-intensity rank,
  then POI-to-built ratio for the industrial/residential split), not left as opaque cluster
  numbers -- so "why is this zone's peer group X?" has a one-line answer.

**Next:** Experience Index and Confidence Score on this real table (`src/compute_scores.py`
currently only runs on synthetic sample data), then Peer Gap -- comparing each zone's Experience
Index against its own peer group's median, which is now possible for the first time.